In [13]:
%load_ext autoreload
%autoreload 2

import time
import numpy as np
import mujoco.viewer
from hexapod_env import HexapodEnv

from stable_baselines3 import PPO
from stable_baselines3.common.monitor import Monitor
from stable_baselines3.common.vec_env import SubprocVecEnv, VecNormalize
from stable_baselines3.common.callbacks import BaseCallback, CheckpointCallback

The autoreload extension is already loaded. To reload it, use:
  %reload_ext autoreload


In [14]:
N_ENVS = 8
TOTAL_STEPS = 3_000_000
RUN_NAME = "stage4.1-collision"
# BASE = "models/stage2-tripod"
env = HexapodEnv()

In [15]:
def make_env():
    return Monitor(HexapodEnv())

vec_env = SubprocVecEnv([make_env for _ in range(N_ENVS)])
vec_env = VecNormalize(vec_env, norm_obs=True, norm_reward=True, clip_obs=10.0)

In [16]:
INFO_KEYS = (
    "forward_vel",
    "side_vel",
    "heading_err_deg",
    "reward",
    "r_vel",
    "r_gait",
    "r_heading",
    "r_vy",
    "r_air",
    "pen_roll",
    "pen_pitch",
    "pen_stroke",
    "pen_angvel",
    "pen_vy",
    "pen_vz",
    "pen_rate",
    "pen_proximity",
    "pen_clearance"
)

class InfoCallback(BaseCallback):
    def _on_step(self):
        for info in self.locals["infos"]:
            for k in INFO_KEYS:
                self.logger.record_mean("env/" + k, info[k])
        return True

In [17]:
model = PPO(
    "MlpPolicy",
    vec_env,
    n_steps=1024,
    batch_size=2048,
    n_epochs=5,
    learning_rate=lambda progress: 3e-5 + (3e-4 - 3e-5) * progress,
    gamma=0.99,
    gae_lambda=0.95,
    clip_range=0.2,
    ent_coef=0.0,
    policy_kwargs=dict(
        net_arch=dict(pi=[256, 256], vf=[256, 256]),
        log_std_init=-1.0,
    ),
    tensorboard_log="logs/",
    device="cpu",
    verbose=1,
)

checkpoint = CheckpointCallback(
    save_freq=100_000 // N_ENVS,
    save_path=f"models/{RUN_NAME}/checkpoints",
    save_vecnormalize=True,
)

Using cpu device


In [18]:
model.learn(total_timesteps=TOTAL_STEPS, callback=[checkpoint, InfoCallback()], tb_log_name=RUN_NAME)

model.save(f"models/{RUN_NAME}/model")
vec_env.save(f"models/{RUN_NAME}/vecnormalize.pkl")
vec_env.close()

Logging to logs/stage4.1-collision_1
height=0.1113, gravity_z=-0.9998, terminated=False, truncated=True
height=0.1118, gravity_z=-0.9998, terminated=False, truncated=True
height=0.1149, gravity_z=-1.0000, terminated=False, truncated=True
height=0.1180, gravity_z=-0.9991, terminated=False, truncated=True
height=0.1125, gravity_z=-0.9998, terminated=False, truncated=True
height=0.1108, gravity_z=-0.9991, terminated=False, truncated=True
height=0.1158, gravity_z=-0.9997, terminated=False, truncated=True
height=0.1135, gravity_z=-0.9989, terminated=False, truncated=True
--------------------------------------
| env/               |               |
|    forward_vel     | -0.00163      |
|    heading_err_deg | 11.245681     |
|    pen_angvel      | 0.606         |
|    pen_clearance   | 1.59          |
|    pen_pitch       | 0.00078332    |
|    pen_proximity   | 0             |
|    pen_rate        | 4.84          |
|    pen_roll        | 0.00047848522 |
|    pen_stroke      | 0.00677       